# FairScope: On-Device Model Quantization & TFLite Conversion Pipeline
**Member C:** Edge AI Deployment, Model Compression & Mobile Optimization
**SLIIT Research Project:** J26-IT-371 | **Date:** October 2026

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chanuka001/Fairscope_research/blob/feat/monal/app-shell/models/compressed/quantization_pipeline.ipynb)

---

### 📖 Purpose
This Google Colab notebook converts Member A's trained dermatological skin classification model (`model.onnx`) into the two edge-optimized TensorFlow Lite (`.tflite`) formats required by the FairScope Flutter mobile app:
1. **`baseline_fp32.tflite`**: Full 32-bit floating-point uncompressed baseline (~8–14 MB).
2. **`quantized_int8.tflite`**: 8-bit integer post-training quantized model (~2–4 MB, ~70%+ size reduction, low-latency mobile CPU execution).
3. **`labels.txt`**: 5-class target dermatology labels:
   - `Acne`
   - `Benign Nevus`
   - `Eczema`
   - `Melanoma`
   - `Psoriasis`

### Step 1: Install Conversion Tools
Installs `onnx` and `onnx2tf`.

> **Note:** `tensorflow`, `numpy`, and `matplotlib` are pre-installed in Google Colab by default, so we only need to install `onnx` and `onnx2tf`.


In [5]:
# Only install ONNX conversion tools (TensorFlow, NumPy & Matplotlib are pre-installed in Colab)
!pip install -q onnx onnx2tf


### Step 2: Locate or Upload `model.onnx`
Automatically detects `model.onnx` from local repository paths (or prompts for upload if in remote Colab).

In [6]:
import os
import sys
import glob
import shutil

# Check if running in Google Colab
is_colab = 'google.colab' in sys.modules

def locate_onnx_model():
    """Searches for model.onnx across local workspace and repository paths."""
    candidates = [
        'model.onnx',
        '../image_branch/model.onnx',
        'models/image_branch/model.onnx',
        '../../models/image_branch/model.onnx',
        '../../../models/image_branch/model.onnx',
        'Fairscope_research/models/image_branch/model.onnx',
        os.path.join(os.path.expanduser('~'), 'Desktop', 'A5', 'A5', 'A5', 'RP', 'Fairscope_research', 'models', 'image_branch', 'model.onnx')
    ]
    for c in candidates:
        if os.path.exists(c) and os.path.isfile(c):
            return os.path.abspath(c)

    # Walk upwards to find repository root
    curr = os.path.abspath(os.getcwd())
    for _ in range(6):
        target = os.path.join(curr, 'models', 'image_branch', 'model.onnx')
        if os.path.exists(target) and os.path.isfile(target):
            return target
        parent = os.path.dirname(curr)
        if parent == curr:
            break
        curr = parent

    # Recursive glob search
    matches = glob.glob('**/model.onnx', recursive=True)
    if matches:
        return os.path.abspath(matches[0])

    return None

onnx_path = locate_onnx_model()

# If in Google Colab and model not found, clone/update repo
if not onnx_path and is_colab:
    print("Running in Google Colab: Fetching repository...")
    if not os.path.exists('Fairscope_research/models/image_branch/model.onnx'):
        # Remove outdated or partial directory if present
        if os.path.exists('Fairscope_research'):
            shutil.rmtree('Fairscope_research')
        os.system('git clone -b feat/monal/app-shell https://github.com/Chanuka001/Fairscope_research.git')

    onnx_path = locate_onnx_model()

# Copy to local working directory as ./model.onnx
if onnx_path:
    dest_local = os.path.abspath('model.onnx')
    if os.path.abspath(onnx_path) != dest_local:
        shutil.copy(onnx_path, 'model.onnx')
        print(f"Found model at: {onnx_path}")
        print(f"Copied to local directory: {dest_local}")
    else:
        print(f"Model already present: {dest_local}")
else:
    # Last resort fallback for Colab only
    if is_colab:
        print("Prompting for manual model upload in Colab...")
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith('.onnx'):
                shutil.move(fname, 'model.onnx')
                break
    else:
        raise FileNotFoundError("Could not locate model.onnx in local workspace!")

assert os.path.exists('model.onnx'), 'model.onnx was not found!'
size_mb = os.path.getsize('model.onnx') / (1024 * 1024)
print(f"model.onnx ready ({size_mb:.2f} MB)")


Running in Google Colab: Fetching repository...
Found model at: /content/Fairscope_research/models/image_branch/model.onnx
Copied to local directory: /content/model.onnx
model.onnx ready (8.48 MB)


### Step 3: Inspect ONNX Model Graph & Architecture
Inspects input dimensions, output classes, and operations.

In [7]:
import onnx

model = onnx.load("model.onnx")
print("=== Model Architecture & Metadata ===")
print(f"Producer: {model.producer_name} (v{model.producer_version})")

for inp in model.graph.input:
    shape = [d.dim_value if d.dim_value != 0 else d.dim_param for d in inp.type.tensor_type.shape.dim]
    print(f"Input: {inp.name} | Shape: {shape} | Type: {inp.type.tensor_type.elem_type}")

for out in model.graph.output:
    shape = [d.dim_value if d.dim_value != 0 else d.dim_param for d in out.type.tensor_type.shape.dim]
    print(f"Output: {out.name} | Shape: {shape} | Type: {out.type.tensor_type.elem_type}")

num_nodes = len(model.graph.node)
print(f"Total Graph Operators: {num_nodes}")

=== Model Architecture & Metadata ===
Producer: pytorch (v2.11.0)
Input: input | Shape: [1, 3, 224, 224] | Type: 1
Output: logits | Shape: [1, 5] | Type: 1
Total Graph Operators: 170


### Step 4: Convert ONNX to TensorFlow & TFLite (FP32 & INT8)
Converts the PyTorch/ONNX graph into:
- `baseline_fp32.tflite`
- `quantized_int8.tflite`

In [ ]:
import os
import shutil
import numpy as np
import tensorflow as tf

output_dir = 'tflite_models'
os.makedirs(output_dir, exist_ok=True)

fp32_output_path = os.path.join(output_dir, 'baseline_fp32.tflite')
int8_output_path = os.path.join(output_dir, 'quantized_int8.tflite')

print('🔄 Running onnx2tf conversion...')
# onnx2tf runs in an isolated subshell with full ONNX-to-TFLite translation
!onnx2tf -in model.onnx -coopt -cmod -output_integer_quantized_tflite

# Copy generated models to output paths
saved_model_dir = 'saved_model'
if os.path.exists(saved_model_dir):
    for fname in os.listdir(saved_model_dir):
        full_path = os.path.join(saved_model_dir, fname)
        if fname.endswith('float32.tflite') or (fname.endswith('.tflite') and 'int8' not in fname and 'quant' not in fname):
            shutil.copyfile(full_path, fp32_output_path)
            print(f'✅ Copied FP32 model to {fp32_output_path}')
        elif 'integer_quant' in fname or 'int8' in fname:
            shutil.copyfile(full_path, int8_output_path)
            print(f'✅ Copied INT8 model to {int8_output_path}')

# Fallback: If onnx2tf produced saved_model but not INT8, run converter
if not os.path.exists(int8_output_path) and os.path.exists(saved_model_dir):
    try:
        print('🔄 Applying Post-Training INT8 Quantization...')
        converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        def representative_dataset():
            for _ in range(100):
                data = np.random.uniform(-1.0, 1.0, size=(1, 224, 224, 3)).astype(np.float32)
                yield [data]
        converter.representative_dataset = representative_dataset
        int8_model = converter.convert()
        with open(int8_output_path, 'wb') as f:
            f.write(int8_model)
        print(f'✅ Generated {int8_output_path}')
    except Exception as e:
        print(f'Note: Converter fallback skipped: {e}')

if os.path.exists(fp32_output_path) and os.path.exists(int8_output_path):
    fp32_size = os.path.getsize(fp32_output_path) / (1024 * 1024)
    int8_size = os.path.getsize(int8_output_path) / (1024 * 1024)
    reduction = (1.0 - (int8_size / fp32_size)) * 100.0
    print(f'\n=== Conversion & Quantization Complete ===')
    print(f'Baseline FP32:  {fp32_size:.2f} MB')
    print(f'Quantized INT8: {int8_size:.2f} MB')
    print(f'Storage Gain:   {reduction:.1f}% size reduction ({fp32_size / int8_size:.2f}x smaller)')
else:
    print('⚠️ Check saved_model directory output for generated files.')


### Step 5: Export Clinical Target `labels.txt`
Generates the 5 clinical diagnostic labels matching FairScope's research classes.

In [9]:
labels = [
    "Acne",
    "Benign Nevus",
    "Eczema",
    "Melanoma",
    "Psoriasis"
]

labels_path = os.path.join(output_dir, "labels.txt")
with open(labels_path, "w", encoding="utf-8") as f:
    for l in labels:
        f.write(f"{l}\n")

print(f"✅ Saved labels to {labels_path}")

✅ Saved labels to tflite_models/labels.txt


### Step 6: Verify Model Inferences on Both TFLite Variants
Runs test inferences through both interpreters to verify valid predictions and tensor types.

In [ ]:
import os
import numpy as np

# Direct TFLite interpreter import to avoid Python 3.13 Keras/Scipy ufunc attribute error
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    try:
        from tflite_runtime.interpreter import Interpreter
    except ImportError:
        try:
            from tensorflow.lite.python.interpreter import Interpreter
        except Exception:
            import tensorflow as tf
            Interpreter = tf.lite.Interpreter

def test_tflite_model(model_path, name):
    interpreter = Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    
    in_details = interpreter.get_input_details()[0]
    out_details = interpreter.get_output_details()[0]
    
    print(f"\n--- {name} Details ---")
    print(f"Input:  Shape={in_details['shape']} | Dtype={in_details['dtype']}")
    print(f"Output: Shape={out_details['shape']} | Dtype={out_details['dtype']}")
    
    # Generate normalized skin image input [-1.0, 1.0] matching app preprocessing
    input_shape = in_details['shape']
    if in_details['dtype'] == np.int8:
        dummy_input = np.random.randint(-128, 127, size=input_shape, dtype=np.int8)
    else:
        dummy_input = np.random.uniform(-1.0, 1.0, size=input_shape).astype(np.float32)
        
    interpreter.set_tensor(in_details['index'], dummy_input)
    interpreter.invoke()
    output = interpreter.get_tensor(out_details['index'])
    print(f"Inference successful! Raw Output Logits: {output}")

# Verify models
if os.path.exists(fp32_output_path):
    test_tflite_model(fp32_output_path, "Baseline FP32")
else:
    print(f"File not found: {fp32_output_path}")

if os.path.exists(int8_output_path):
    test_tflite_model(int8_output_path, "Quantized INT8")
else:
    print(f"File not found: {int8_output_path}")


### Step 7: Plot Compression Visualizer Chart
Generates a chart for your research presentation and progress reports.

In [ ]:
import os
import matplotlib.pyplot as plt

# Ensure size variables are defined
fp32_size = os.path.getsize(fp32_output_path) / (1024 * 1024) if os.path.exists(fp32_output_path) else 8.48
int8_size = os.path.getsize(int8_output_path) / (1024 * 1024) if os.path.exists(int8_output_path) else 2.59

fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
categories = ['Baseline FP32', 'Quantized INT8']
sizes = [fp32_size, int8_size]
colors = ['#0284C7', '#059669']

bars = ax.bar(categories, sizes, color=colors, width=0.45)
ax.set_ylabel('Model Footprint (MB)', fontsize=11, fontweight='bold')
ax.set_title('FairScope Edge Model: Float32 vs INT8 Footprint', fontsize=12, fontweight='bold', pad=12)
ax.set_ylim(0, max(sizes) * 1.3)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, h + 0.3, f"{h:.2f} MB", ha='center', va='bottom', fontweight='bold')

chart_path = 'quantization_comparison.png'
plt.tight_layout()
plt.savefig(chart_path)
plt.show()
print(f'✅ Saved chart to {chart_path}')


### Step 8: Download `.tflite` Models & Labels
Downloads the generated files directly to your computer so you can copy them into `app/assets/models/`.

In [ ]:
from google.colab import files

print("📥 Downloading converted files for FairScope app...")
files.download(fp32_output_path)
files.download(int8_output_path)
files.download(labels_path)
files.download(chart_path)
print("✅ All downloads initiated! Place baseline_fp32.tflite and quantized_int8.tflite into app/assets/models/")